# EDA – gresspollen

- undersøke manglende verdier
- se på fordelingen av gresspollen
- undersøke hvordan gresspollen varierer over tid og mellom år
- se på sammenhenger mellom pollen og værvariabler

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


 1. Leser inn datasettet:

In [ ]:
df = pd.read_csv(
    "../data/full_pollen_and_weather_data_Zurich.csv",
    sep=";"
)

df.shape

(32352, 46)

2. Gir kolonnene mer forståelige navn og velger relevante variabler:

In [ ]:
df = df.rename(columns={
    "station_abbr_pollen": "Pollenstasjon",
    "station_abbr_weather": "Værstasjon",
    "master_timestamp": "Tid",
    "timestamp_pollen": "Pollen_tid",
    "timestamp_weather": "Vær_tid",
    "kabetuh0": "Bjørk",
    "khpoach0": "Gress",
    "tre200h0": "Lufttemperatur",
    "fkl010h0": "Vindstyrke",
    "ure200h0": "Luftfuktighet",
    "rre150h0": "Nedbør",
    "sre000h0": "Solskinn",
    "dkl010h0": "Vindretning"
})

data = df[[
    "Tid",
    "Gress",
    "Lufttemperatur",
    "Vindstyrke",
    "Luftfuktighet",
    "Solskinn",
    "Nedbør"
]].copy()


3. Undersøker rådata:

In [ ]:
print("Størrelse:", data.shape)
print("\nDatatyper:")
print(data.dtypes)

print("\nManglende verdier:")
print(data.isna().sum())

print("\nAntall duplikater:")
print(data.duplicated().sum())

Størrelse: (32352, 7)

Datatyper:
Tid                   str
Bjørk             float64
Lufttemperatur    float64
Vindstyrke        float64
Luftfuktighet     float64
Solskinn            int64
Nedbør            float64
dtype: object

Manglende verdier:
Tid                 0
Bjørk             779
Lufttemperatur      0
Vindstyrke          7
Luftfuktighet       0
Solskinn            0
Nedbør              0
dtype: int64

Antall duplikater:
0


4. Konverterer tidspunkt og lager daglige verdier:

In [ ]:
data["Tid"] = pd.to_datetime(data["Tid"])
data["dato"] = data["Tid"].dt.date

daglig_alle = data.groupby("dato").agg(
    gress_mean=("Gress", "mean"),
    gress_antall=("Gress", "count"),

    temperatur_mean=("Lufttemperatur", "mean"),
    temperatur_antall=("Lufttemperatur", "count"),

    vind_mean=("Vindstyrke", "mean"),
    vind_antall=("Vindstyrke", "count"),

    luftfuktighet_mean=("Luftfuktighet", "mean"),
    luftfuktighet_antall=("Luftfuktighet", "count"),

    solskinn_total=("Solskinn", "sum"),
    solskinn_antall=("Solskinn", "count"),
    
    nedbør_total=("Nedbør", "sum"),
    nedbør_antall=("Nedbør", "count")
).reset_index()

daglig_alle["dato"] = pd.to_datetime(daglig_alle["dato"])
daglig_alle["dag_i_året"] = daglig_alle["dato"].dt.dayofyear


5. Filtrerer dager med for få målinger:

In [ ]:
daglig_gyldige = daglig_alle[
    (daglig_alle["gress_antall"] >= 20) &
    (daglig_alle["temperatur_antall"] >= 20) &
    (daglig_alle["vind_antall"] >= 20) &
    (daglig_alle["luftfuktighet_antall"] >= 20) &
    (daglig_alle["solskinn_antall"] >= 20) &
    (daglig_alle["nedbør_antall"] >= 20)
].copy()

daglig_gyldige = daglig_gyldige.sort_values("dato").copy()

print("Før filtrering:", daglig_alle.shape)
print("Etter filtrering:", daglig_gyldige.shape)


6. Sjekker manglende verdier etter cleaning:

In [ ]:
daglig_gyldige.isna().sum()

7. Statistikk:

In [ ]:
daglig_gyldige[[
    "gress_mean",
    "temperatur_mean",
    "vind_mean",
    "luftfuktighet_mean",
    "solskinn_total",
    "nedbør_total"
]].describe()


8. Gresspollen over tid:

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(daglig_gyldige["dato"], daglig_gyldige["gress_mean"])
plt.xlabel("Dato")
plt.ylabel("Gjennomsnittlig gresspollenkonsentrasjon")
plt.title("Daglig gresspollenkonsentrasjon i Zürich")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


9. Gresspollen i et spesifikt år:


In [ ]:
år = 2024

data_år = daglig_gyldige[
    daglig_gyldige["dato"].dt.year == år
].copy()

plt.figure(figsize=(12, 6))

plt.plot(
    data_år["dato"],
    data_år["gress_mean"]
)

plt.xlabel("Dato")
plt.ylabel("Gjennomsnittlig gresskonsentrasjon")
plt.title(f"Gress i Zürich - {år}")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


10. Sammenligner gresspollen mellom år:


In [ ]:
daglig_gyldige["år"] = daglig_gyldige["dato"].dt.year

gress_per_år = daglig_gyldige.groupby("år")["gress_mean"].agg(
    ["mean", "median", "max"]
)

gress_per_år


11. Fordeling av gresspollen:


In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(daglig_gyldige["gress_mean"].dropna(), bins=40)
plt.xlabel("Gjennomsnittlig gresspollenkonsentrasjon")
plt.ylabel("Antall dager")
plt.title("Fordeling av daglige gresspollenverdier")
plt.tight_layout()
plt.show()


12. Korrelasjon mellom pollen og værvariabler:


In [ ]:
korrelasjon = daglig_gyldige[[
    "gress_mean",
    "temperatur_mean",
    "vind_mean",
    "luftfuktighet_mean",
    "solskinn_total",
    "nedbør_total"
]].corr()

korrelasjon


13. Gresspollen mot temperatur:


In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(
    daglig_gyldige["temperatur_mean"],
    daglig_gyldige["gress_mean"],
    alpha=0.5
)
plt.xlabel("Gjennomsnittlig temperatur")
plt.ylabel("Gjennomsnittlig gresspollenkonsentrasjon")
plt.title("Gresspollen og temperatur")
plt.tight_layout()
plt.show()


14. Gresspollen mot nedbør:

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(
    daglig_gyldige["nedbør_total"],
    daglig_gyldige["gress_mean"],
    alpha=0.5
)
plt.xlabel("Daglig nedbør")
plt.ylabel("Gjennomsnittlig gresspollenkonsentrasjon")
plt.title("Gresspollen og nedbør")
plt.tight_layout()
plt.show()
